In [5]:
import copy
import math

import numpy as np

In [6]:
MEASUREMENTS = {
    "base_to_leg_front": 0.075,
    "base_to_leg_back": 0.075,
    "base_to_leg_left": 0.0445,
    "base_to_leg_right": 0.0335,
    "trans_1_2": [0, 0, -0.039],
    "trans_2_3": [0, -0.0494, 0.0685],
    "trans_3_ee": [0.06231, -0.06216, -0.018],
}

In [7]:
def forward_kinematics(
    theta: list[float], measurements: dict[str, float | list[float]]
) -> list[float]:
    """
    thetas: dict of leg_id to joint angles (_1, _2, _3) for the leg.
    """

    def rotation_x(angle):
        # rotation about the x-axis implemented for you
        return np.array(
            [
                [1, 0, 0, 0],
                [0, np.cos(angle), -np.sin(angle), 0],
                [0, np.sin(angle), np.cos(angle), 0],
                [0, 0, 0, 1],
            ]
        )

    def rotation_y(angle):
        return np.array(
            [
                [np.cos(angle), 0, np.sin(angle), 0],
                [0, 1, 0, 0],
                [-np.sin(angle), 0, np.cos(angle), 0],
                [0, 0, 0, 1],
            ]
        )

    def rotation_z(angle):
        return np.array(
            [
                [np.cos(angle), -np.sin(angle), 0, 0],
                [np.sin(angle), np.cos(angle), 0, 0],
                [0, 0, 1, 0],
                [0, 0, 0, 1],
            ]
        )

    def translation(x, y, z):
        return np.array(
            [
                [1, 0, 0, x],
                [0, 1, 0, y],
                [0, 0, 1, z],
                [0, 0, 0, 1],
            ]
        )

    base_to_leg_front = measurements["base_to_leg_front"]
    base_to_leg_left = measurements["base_to_leg_left"]

    trans_1_2 = translation(*measurements["trans_1_2"])
    trans_2_3 = translation(*measurements["trans_2_3"])
    trans_3_ee = translation(*measurements["trans_3_ee"])

    rot_0_1 = rotation_x(1.57080)
    rot_1_2 = rotation_y(-1.57080)
    rot_2_3 = rotation_y(1.57080)

    # theta is positive when the Z-axis points out of the BOTTOM (i.e. uncovered metal parts) of the BLDC motor.
    # Since we keep the frame orientation the same on both left and right sides, motors 1 and 3 will use negative angles on the left and positive angles on the right.

    T_0_1 = (
        translation(base_to_leg_front, base_to_leg_left, 0)
        @ rot_0_1
        @ rotation_z(-theta[0])
    )
    T_1_2 = trans_1_2 @ rot_1_2 @ rotation_z(+theta[1])
    T_2_3 = trans_2_3 @ rot_2_3 @ rotation_z(-theta[2])
    T_3_ee = trans_3_ee
    T_0_ee = T_0_1 @ T_1_2 @ T_2_3 @ T_3_ee

    return T_0_ee[:3, 3].copy()

In [9]:
forward_kinematics([0.0, 0.0, 0.0], MEASUREMENTS)

array([ 0.06881   ,  0.10150066, -0.11155979])

In [10]:
simulated_inputs = {
    "theta_degs": [0.0, 20.0, 45.0, 70.0],
    "l1_l2_err_cm": [0.2, 0.4, 0.8, 1.0],
}

In [11]:
# Compute errors for front left leg
def simulate_errors(inputs: dict[str, list[float]]) -> dict[str, list[float]]:
    """Return error in EE position for each dimension"""
    out = {}
    measurements = copy.deepcopy(MEASUREMENTS)
    for deg in inputs["theta_degs"]:
        theta = [math.radians(deg) for _ in range(3)]
        ref_ee = forward_kinematics(theta, MEASUREMENTS)
        for err_cm in inputs["l1_l2_err_cm"]:
            measurements["trans_1_2"][2] = measurements["trans_1_2"][2] + err_cm / 100.0
            ee = forward_kinematics(theta, measurements)

            diff = ee - ref_ee
            out[(deg, err_cm)] = np.round(diff, 2)

    return out

In [12]:
errs = simulate_errors(simulated_inputs)

In [13]:
errs

{(0.0, 0.2): array([ 0., -0., -0.]),
 (0.0, 0.4): array([ 0.  , -0.01, -0.  ]),
 (0.0, 0.8): array([ 0.  , -0.01, -0.  ]),
 (0.0, 1.0): array([ 0.  , -0.02, -0.  ]),
 (20.0, 0.2): array([ 0.  , -0.03, -0.  ]),
 (20.0, 0.4): array([ 0.  , -0.03, -0.  ]),
 (20.0, 0.8): array([ 0.  , -0.04, -0.  ]),
 (20.0, 1.0): array([ 0.  , -0.05, -0.  ]),
 (45.0, 0.2): array([ 0.  , -0.05, -0.  ]),
 (45.0, 0.4): array([ 0.  , -0.05, -0.  ]),
 (45.0, 0.8): array([ 0.  , -0.06, -0.  ]),
 (45.0, 1.0): array([ 0.  , -0.07, -0.  ]),
 (70.0, 0.2): array([ 0.  , -0.07, -0.  ]),
 (70.0, 0.4): array([ 0.  , -0.08, -0.  ]),
 (70.0, 0.8): array([ 0.  , -0.09, -0.  ]),
 (70.0, 1.0): array([ 0. , -0.1, -0. ])}